# TFR pipeline: только `lib.*`

Используется функциональность из пакета `lib` (без `utils.*`):

- `lib.data.TFRDataset`, `lib.training.epochs`
- `lib.models` — AlexNet и TFR-Transformer
- `lib.optuna` — `make_objective_engine`, фабрики сплитов и параметров
- `lib.modes.offline_tfr_supervised` + `lib.core.pipeline.PipelineRunner`

**Запуск:** рабочая директория — папка `notebooks/`, корень проекта `NeuronDeCo` = `Path.cwd().parent`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import optuna
import torch
from torch.utils.data import DataLoader

root = Path.cwd().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from lib.core.context import RunContext
from lib.core.pipeline import PipelineRunner
from lib.data import TFRDataset
from lib.models.alexnet import AlexNetTFR
from lib.models.tfr_transformer import TFRTransformerWrapper
from lib.modes.offline_tfr_supervised import OfflineTFRSupervisedMode
from lib.optuna import (
    attrs_fn,
    loss_slope,
    make_objective_engine,
    make_splits_fn_factory,
    objectives_fn,
    params_fn_factory,
    params_fn_factory_transformer,
    run_fold_fn_factory,
)
from lib.registry import MODE_REGISTRY
from lib.training.epochs import eval_one_epoch_f1_macro, train_one_epoch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("project root:", root)
print("device:", device)

## 1. Синтетические `X (N,C,F,T)`, `y (N,)`

In [ ]:
rng = np.random.default_rng(0)
n, c, f, t = 96, 7, 80, 64
X = rng.standard_normal((n, c, f, t)).astype(np.float32)
y = rng.integers(0, 2, size=(n,), dtype=np.int64)
X.shape, y.shape

## 2. Optuna (AlexNet): `study.optimize`

In [ ]:
seed = 42
objective_alex = make_objective_engine(
    X=X,
    y=y,
    make_splits_fn=make_splits_fn_factory(test_size=0.25, seed=seed, cv=False),
    run_fold_fn=run_fold_fn_factory(
        ModelCls=AlexNetTFR,
        device=device,
        max_epochs=2,
        patience=1,
        TFRDataset=TFRDataset,
        DataLoader=DataLoader,
        train_one_epoch=train_one_epoch,
        eval_one_epoch_f1_macro=eval_one_epoch_f1_macro,
        loss_metric=loss_slope,
    ),
    aggregate_mode="median",
    params_fn=params_fn_factory(in_channels=c, num_classes=2),
    objectives_fn=objectives_fn,
    attrs_fn=attrs_fn,
)
study_a = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.TPESampler(seed=seed),
)
study_a.optimize(objective_alex, n_trials=2, show_progress_bar=True)
study_a.trials[-1].values, study_a.trials[-1].params

## 3. Optuna (Transformer)

In [ ]:
objective_tr = make_objective_engine(
    X=X,
    y=y,
    make_splits_fn=make_splits_fn_factory(test_size=0.25, seed=seed, cv=False),
    run_fold_fn=run_fold_fn_factory(
        ModelCls=TFRTransformerWrapper,
        device=device,
        max_epochs=2,
        patience=1,
        TFRDataset=TFRDataset,
        DataLoader=DataLoader,
        train_one_epoch=train_one_epoch,
        eval_one_epoch_f1_macro=eval_one_epoch_f1_macro,
        loss_metric=loss_slope,
    ),
    aggregate_mode="median",
    params_fn=params_fn_factory_transformer(
        num_classes=2,
        seq_len=t,
        batch_size_choices=(8, 16),
        embed_dim_choices=(32, 64),
        dim_fc_choices=(64, 128),
    ),
    objectives_fn=objectives_fn,
    attrs_fn=attrs_fn,
)
study_t = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.TPESampler(seed=seed + 1),
)
study_t.optimize(objective_tr, n_trials=2, show_progress_bar=True)
study_t.trials[-1].values, study_t.trials[-1].params

## 4. Offline supervised (`PipelineRunner` + `offline_tfr_supervised`)

In [ ]:
model = AlexNetTFR(in_channels=c, num_classes=2)
mode = MODE_REGISTRY["offline_tfr_supervised"]()
ctx = RunContext(
    run_id="nb-offline",
    device=str(device),
    seed=seed,
    metadata={
        "max_epochs": 4,
        "patience": 2,
        "batch_size": 16,
        "lr": 1e-3,
        "test_size": 0.25,
    },
)
result = PipelineRunner(mode, model, ctx).run(X, y)
result